In [1]:
# Диагностика проблем с созданием модели Llama 3.2 3B в HookedTransformer
# Этот notebook поможет понять, что происходит при создании модели

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer, HookedTransformerConfig
from typing import Dict, Any
import json

# Модель из конфигурации try.yaml
model_name = "ExplosionNuclear/Llama-2.3-3B-Instruct-special-merged-with-19-exp"

print(f"Загружаем модель: {model_name}")
print("="*50)


Загружаем модель: ExplosionNuclear/Llama-2.3-3B-Instruct-special-merged-with-19-exp


In [2]:
# Шаг 1: Загружаем оригинальную HuggingFace модель и изучаем её конфигурацию
print("1. Загружаем HuggingFace модель...")

hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # Используем float32 как в конфигурации
    device_map="cpu"  # Загружаем на CPU для анализа
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_config = hf_model.config

print("Конфигурация HuggingFace модели:")
print(f"  - num_hidden_layers: {hf_config.num_hidden_layers}")
print(f"  - hidden_size (d_model): {hf_config.hidden_size}")
print(f"  - num_attention_heads: {hf_config.num_attention_heads}")
print(f"  - intermediate_size (d_mlp): {hf_config.intermediate_size}")
print(f"  - vocab_size: {hf_config.vocab_size}")
print(f"  - max_position_embeddings: {hf_config.max_position_embeddings}")
print(f"  - hidden_act: {hf_config.hidden_act}")
print(f"  - model_type: {hf_config.model_type}")

# Проверяем d_head расчет
calculated_d_head = hf_config.hidden_size // hf_config.num_attention_heads
print(f"  - calculated d_head: {calculated_d_head}")

print(f"\nИз try.yaml ожидается d_in = 3072")
print(f"Фактический hidden_size = {hf_config.hidden_size}")
print(f"Совпадает: {hf_config.hidden_size == 3072}")

print("\nПолная конфигурация модели:")
print(json.dumps(hf_config.to_dict(), indent=2))


1. Загружаем HuggingFace модель...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Конфигурация HuggingFace модели:
  - num_hidden_layers: 28
  - hidden_size (d_model): 3072
  - num_attention_heads: 24
  - intermediate_size (d_mlp): 8192
  - vocab_size: 128258
  - max_position_embeddings: 131072
  - hidden_act: silu
  - model_type: llama
  - calculated d_head: 128

Из try.yaml ожидается d_in = 3072
Фактический hidden_size = 3072
Совпадает: True

Полная конфигурация модели:
{
  "vocab_size": 128258,
  "max_position_embeddings": 131072,
  "hidden_size": 3072,
  "intermediate_size": 8192,
  "num_hidden_layers": 28,
  "num_attention_heads": 24,
  "num_key_value_heads": 8,
  "hidden_act": "silu",
  "initializer_range": 0.02,
  "rms_norm_eps": 1e-05,
  "pretraining_tp": 1,
  "use_cache": true,
  "rope_theta": 500000.0,
  "rope_scaling": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "attention_bias": false,
  "attention_dropout": 0.0,
  "mlp_bias": false,
  "head_

In [3]:
# Шаг 2: Создаем HookedTransformerConfig как в текущем коде
print("2. Создаем HookedTransformerConfig (текущий метод)...")

max_ctx = min(hf_config.max_position_embeddings, 2048)

# Текущая конфигурация из utils.py
current_cfg = HookedTransformerConfig(
    n_layers=hf_config.num_hidden_layers,
    d_model=hf_config.hidden_size,
    d_head=hf_config.hidden_size // hf_config.num_attention_heads,
    n_heads=hf_config.num_attention_heads,
    d_mlp=hf_config.intermediate_size,
    d_vocab=hf_config.vocab_size,
    n_ctx=max_ctx,
    act_fn=hf_config.hidden_act,
    model_name=model_name,
    normalization_type="RMS",
    device="cpu",
    use_hook_mlp_in=True,
)

print("Конфигурация HookedTransformer (текущая):")
print(f"  - n_layers: {current_cfg.n_layers}")
print(f"  - d_model: {current_cfg.d_model}")
print(f"  - d_head: {current_cfg.d_head}")
print(f"  - n_heads: {current_cfg.n_heads}")
print(f"  - d_mlp: {current_cfg.d_mlp}")
print(f"  - d_vocab: {current_cfg.d_vocab}")
print(f"  - n_ctx: {current_cfg.n_ctx}")
print(f"  - act_fn: {current_cfg.act_fn}")
print(f"  - normalization_type: {current_cfg.normalization_type}")

# Проверяем что все параметры корректны
print(f"\nПроверки:")
print(f"  - d_model * n_heads == d_head * n_heads: {current_cfg.d_model == current_cfg.d_head * current_cfg.n_heads}")
print(f"  - act_fn соответствует: {current_cfg.act_fn == hf_config.hidden_act}")
print(f"  - vocab размеры: {current_cfg.d_vocab == hf_config.vocab_size}")


2. Создаем HookedTransformerConfig (текущий метод)...
Конфигурация HookedTransformer (текущая):
  - n_layers: 28
  - d_model: 3072
  - d_head: 128
  - n_heads: 24
  - d_mlp: 8192
  - d_vocab: 128258
  - n_ctx: 2048
  - act_fn: silu
  - normalization_type: RMS

Проверки:
  - d_model * n_heads == d_head * n_heads: True
  - act_fn соответствует: True
  - vocab размеры: True


In [4]:
# Шаг 3: Создаем HookedTransformer и пытаемся загрузить веса
print("3. Создаем HookedTransformer и загружаем веса...")

try:
    hooked_model = HookedTransformer(current_cfg)
    print("✓ HookedTransformer создан успешно")
    
    # Проверяем архитектуру модели
    print(f"Архитектура HookedTransformer:")
    print(f"  - Общее количество параметров: {sum(p.numel() for p in hooked_model.parameters()):,}")
    print(f"  - Embedding размер: {hooked_model.embed.W_E.shape}")
    print(f"  - Позиционные embeddings: {hooked_model.pos_embed.W_pos.shape}")
    print(f"  - Количество слоев: {len(hooked_model.blocks)}")
    
    if len(hooked_model.blocks) > 0:
        first_block = hooked_model.blocks[0]
        print(f"  - Первый блок attn W_Q: {first_block.attn.W_Q.shape}")
        print(f"  - Первый блок attn W_K: {first_block.attn.W_K.shape}")
        print(f"  - Первый блок attn W_V: {first_block.attn.W_V.shape}")
        print(f"  - Первый блок attn W_O: {first_block.attn.W_O.shape}")
        print(f"  - Первый блок MLP W_in: {first_block.mlp.W_in.shape}")
        print(f"  - Первый блок MLP W_out: {first_block.mlp.W_out.shape}")
    
except Exception as e:
    print(f"✗ Ошибка при создании HookedTransformer: {e}")
    print(f"Тип ошибки: {type(e)}")
    import traceback
    traceback.print_exc()


3. Создаем HookedTransformer и загружаем веса...
✓ HookedTransformer создан успешно
Архитектура HookedTransformer:
  - Общее количество параметров: 3,261,522,178
  - Embedding размер: torch.Size([128258, 3072])
  - Позиционные embeddings: torch.Size([2048, 3072])
  - Количество слоев: 28
  - Первый блок attn W_Q: torch.Size([24, 3072, 128])
  - Первый блок attn W_K: torch.Size([24, 3072, 128])
  - Первый блок attn W_V: torch.Size([24, 3072, 128])
  - Первый блок attn W_O: torch.Size([24, 128, 3072])
  - Первый блок MLP W_in: torch.Size([3072, 8192])
  - Первый блок MLP W_out: torch.Size([8192, 3072])


In [6]:
# Шаг 4: Пытаемся загрузить веса и проверяем совместимость
print("4. Загружаем веса из HuggingFace модели...")

try:
    # Проверяем архитектуру HF модели
    print("Архитектура HuggingFace модели:")
    print(f"  - Общее количество параметров: {sum(p.numel() for p in hf_model.parameters()):,}")
    
    # Получаем state_dict от HF модели
    hf_state_dict = hf_model.state_dict()
    hooked_state_dict = hooked_model.state_dict()
    
    print(f"\nКлючи в HF state_dict (первые 10):")
    for i, key in enumerate(list(hf_state_dict.keys())[:10]):
        print(f"  {i+1}. {key}: {hf_state_dict[key].shape}")
    
    print(f"\nКлючи в HookedTransformer state_dict (первые 10):")
    for i, key in enumerate(list(hooked_state_dict.keys())[:10]):
        print(f"  {i+1}. {key}: {hooked_state_dict[key].shape}")
    
    # Пытаемся загрузить веса
    print(f"\n5. Пытаемся загрузить веса...")
    missing_keys, unexpected_keys = hooked_model.load_state_dict(hf_state_dict, strict=False)
    
    print(f"✓ Веса загружены с strict=False")
    print(f"  - Отсутствующие ключи: {len(missing_keys)}")
    print(f"  - Неожиданные ключи: {len(unexpected_keys)}")
    
    if missing_keys:
        print(f"\nПервые 5 отсутствующих ключей:")
        for key in missing_keys[:5]:
            print(f"  - {key}")
    
    if unexpected_keys:
        print(f"\nПервые 5 неожиданных ключей:")
        for key in unexpected_keys[:5]:
            print(f"  - {key}")
            
except Exception as e:
    print(f"✗ Ошибка при загрузке весов: {e}")
    import traceback
    traceback.print_exc()


4. Загружаем веса из HuggingFace модели...
Архитектура HuggingFace модели:
  - Общее количество параметров: 3,212,755,968

Ключи в HF state_dict (первые 10):
  1. model.embed_tokens.weight: torch.Size([128258, 3072])
  2. model.layers.0.self_attn.q_proj.weight: torch.Size([3072, 3072])
  3. model.layers.0.self_attn.k_proj.weight: torch.Size([1024, 3072])
  4. model.layers.0.self_attn.v_proj.weight: torch.Size([1024, 3072])
  5. model.layers.0.self_attn.o_proj.weight: torch.Size([3072, 3072])
  6. model.layers.0.mlp.gate_proj.weight: torch.Size([8192, 3072])
  7. model.layers.0.mlp.up_proj.weight: torch.Size([8192, 3072])
  8. model.layers.0.mlp.down_proj.weight: torch.Size([3072, 8192])
  9. model.layers.0.input_layernorm.weight: torch.Size([3072])
  10. model.layers.0.post_attention_layernorm.weight: torch.Size([3072])

Ключи в HookedTransformer state_dict (первые 10):
  1. embed.W_E: torch.Size([128258, 3072])
  2. pos_embed.W_pos: torch.Size([2048, 3072])
  3. blocks.0.ln1.w: torch.

In [11]:
def map_hf_to_hooked_state_dict(hf_state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    """
    Простая функция которая переименовывает ключи HuggingFace в ключи HookedTransformer
    """
    import re
    mapped_dict = {}
    
    for hf_key, weight in hf_state_dict.items():
        hooked_key = None
        
        # Embeddings
        if hf_key == "model.embed_tokens.weight":
            hooked_key = "embed.W_E"
        
        # Final layer norm
        elif hf_key == "model.norm.weight":
            hooked_key = "ln_final.w"
            
        # LM head
        elif hf_key == "lm_head.weight":
            hooked_key = "unembed.W_U"
            
        # Layer-specific mappings
        elif hf_key.startswith("model.layers."):
            # Извлекаем номер слоя
            layer_match = re.match(r"model\.layers\.(\d+)\.(.*)", hf_key)
            if layer_match:
                layer_num = layer_match.group(1)
                rest = layer_match.group(2)
                
                # Attention weights
                if rest == "self_attn.q_proj.weight":
                    hooked_key = f"blocks.{layer_num}.attn.W_Q"
                elif rest == "self_attn.k_proj.weight":
                    hooked_key = f"blocks.{layer_num}.attn.W_K"
                elif rest == "self_attn.v_proj.weight":
                    hooked_key = f"blocks.{layer_num}.attn.W_V"
                elif rest == "self_attn.o_proj.weight":
                    hooked_key = f"blocks.{layer_num}.attn.W_O"
                
                # MLP weights
                elif rest == "mlp.gate_proj.weight":
                    hooked_key = f"blocks.{layer_num}.mlp.W_in"
                elif rest == "mlp.up_proj.weight":
                    hooked_key = f"blocks.{layer_num}.mlp.W_gate"
                elif rest == "mlp.down_proj.weight":
                    hooked_key = f"blocks.{layer_num}.mlp.W_out"
                
                # Layer norms
                elif rest == "input_layernorm.weight":
                    hooked_key = f"blocks.{layer_num}.ln1.w"
                elif rest == "post_attention_layernorm.weight":
                    hooked_key = f"blocks.{layer_num}.ln2.w"
        
        # Если нашли маппинг, добавляем в результат
        if hooked_key:
            mapped_dict[hooked_key] = weight
    
    return mapped_dict

def get_custom_hf_model_simple(hf_model, tokenizer, kwargs: Dict[str, Any] = {}) -> HookedTransformer:
    hf_config = hf_model.config
    
    # Создаем конфигурацию HookedTransformer
    max_ctx = min(hf_config.max_position_embeddings, 2048)
    d_head = hf_config.hidden_size // hf_config.num_attention_heads
    
    cfg = HookedTransformerConfig(
        n_layers=hf_config.num_hidden_layers,
        d_model=hf_config.hidden_size,
        d_head=d_head,
        n_heads=hf_config.num_attention_heads,
        d_mlp=hf_config.intermediate_size,
        d_vocab=hf_config.vocab_size,
        n_ctx=max_ctx,
        act_fn=hf_config.hidden_act,
        model_name=model_name,
        normalization_type="RMS",
        device="cpu",
        use_hook_mlp_in=True,
        eps=getattr(hf_config, 'rms_norm_eps', 1e-5),
        gated_mlp=True,
        final_rms=True,
    )
    
    hooked_model = HookedTransformer(cfg)
    
    # Простое решение: переименовываем ключи и загружаем
    hf_state_dict = hf_model.state_dict()
    mapped_state_dict = map_hf_to_hooked_state_dict(hf_state_dict)
    
    print(f"Маппинг создал {len(mapped_state_dict)} ключей")
    
    # Загружаем с автоматическим определением отсутствующих весов
    missing_keys, unexpected_keys = hooked_model.load_state_dict(mapped_state_dict, strict=False)
    
    print(f"Отсутствующие ключи: {len(missing_keys)}")
    print(f"Неожиданные ключи: {len(unexpected_keys)}")
    
    if missing_keys:
        print("Первые 5 отсутствующих:")
        for key in missing_keys[:5]:
            print(f"  - {key}")
    
    hooked_model.set_tokenizer(tokenizer)
    return hooked_model



hooked_model = get_custom_hf_model_simple(hf_model, tokenizer)

Маппинг создал 255 ключей


RuntimeError: Error(s) in loading state_dict for HookedTransformer:
	size mismatch for blocks.0.attn.W_Q: copying a param with shape torch.Size([3072, 3072]) from checkpoint, the shape in current model is torch.Size([24, 3072, 128]).
	size mismatch for blocks.0.attn.W_O: copying a param with shape torch.Size([3072, 3072]) from checkpoint, the shape in current model is torch.Size([24, 128, 3072]).
	size mismatch for blocks.0.attn.W_K: copying a param with shape torch.Size([1024, 3072]) from checkpoint, the shape in current model is torch.Size([24, 3072, 128]).
	size mismatch for blocks.0.attn.W_V: copying a param with shape torch.Size([1024, 3072]) from checkpoint, the shape in current model is torch.Size([24, 3072, 128]).
	size mismatch for blocks.0.mlp.W_in: copying a param with shape torch.Size([8192, 3072]) from checkpoint, the shape in current model is torch.Size([3072, 8192]).
	size mismatch for blocks.0.mlp.W_gate: copying a param with shape torch.Size([8192, 3072]) from checkpoi

In [20]:
def get_custom_hf_model(model_name: str, kwargs: Dict[str, Any] = {}) -> HookedTransformer:
    """
    Напрямую используем convert_llama_weights, обходя проверку официальных моделей
    """
    print(f"Загружаем модель с прямой конвертацией: {model_name}")
    
    # Загружаем HF модель
    hf_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="cpu",
        **kwargs
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    hf_config = hf_model.config
    
    print(f"HF конфигурация:")
    print(f"  - hidden_size: {hf_config.hidden_size}")
    print(f"  - num_attention_heads: {hf_config.num_attention_heads}")
    print(f"  - intermediate_size: {hf_config.intermediate_size}")
    print(f"  - num_hidden_layers: {hf_config.num_hidden_layers}")
    print(f"  - vocab_size: {hf_config.vocab_size}")
    
    # Создаем конфигурацию как для Llama
    cfg_dict = {
        "d_model": hf_config.hidden_size,
        "d_head": hf_config.hidden_size // hf_config.num_attention_heads,
        "n_heads": hf_config.num_attention_heads,
        "d_mlp": hf_config.intermediate_size,
        "n_layers": hf_config.num_hidden_layers,
        "n_ctx": min(hf_config.max_position_embeddings, 2048),
        "eps": getattr(hf_config, 'rms_norm_eps', 1e-6),
        "d_vocab": hf_config.vocab_size,
        "act_fn": hf_config.hidden_act,
        "normalization_type": "RMS",
        "positional_embedding_type": "rotary",
        "rotary_adjacent_pairs": False,
        "rotary_dim": hf_config.hidden_size // hf_config.num_attention_heads,
        "final_rms": True,
        "gated_mlp": True,
        "model_name": model_name.split("/")[-1],
        "init_weights": False,
        "device": "cpu",
        "dtype": torch.float32,
    }
    
    # Добавляем n_key_value_heads если есть
    if hasattr(hf_config, 'num_key_value_heads') and hf_config.num_key_value_heads != hf_config.num_attention_heads:
        cfg_dict["n_key_value_heads"] = hf_config.num_key_value_heads
    
    if hasattr(hf_config, 'rope_theta'):
        cfg_dict["rotary_base"] = hf_config.rope_theta
        print(f"✅ Установлен rotary_base = {hf_config.rope_theta}")
    
    
    cfg = HookedTransformerConfig.from_dict(cfg_dict)
    
    print(f"HookedTransformer конфигурация:")
    print(f"  - d_model: {cfg.d_model}")
    print(f"  - n_layers: {cfg.n_layers}")
    print(f"  - n_heads: {cfg.n_heads}")
    print(f"  - d_mlp: {cfg.d_mlp}")
    print(f"  - gated_mlp: {cfg.gated_mlp}")
    
    # Импортируем и используем convert_llama_weights напрямую
    from transformer_lens.loading_from_pretrained import convert_llama_weights
    
    print("Конвертируем веса через convert_llama_weights...")
    
    # Отключаем градиенты для HF модели
    for param in hf_model.parameters():
        param.requires_grad = False
    
    # Конвертируем веса
    state_dict = convert_llama_weights(hf_model, cfg)
    
    print(f"✓ Веса сконвертированы! Количество ключей: {len(state_dict)}")
    print(f"Примеры ключей:")
    for i, key in enumerate(list(state_dict.keys())[:5]):
        print(f"  {i+1}. {key}: {state_dict[key].shape}")
    
    # Создаем HookedTransformer
    model = HookedTransformer(cfg)
    
    # Загружаем веса
    missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
    
    print(f"Загрузка весов:")
    print(f"  - Отсутствующие ключи: {len(missing_keys)}")
    print(f"  - Неожиданные ключи: {len(unexpected_keys)}")
    
    if missing_keys:
        print(f"Первые 5 отсутствующих ключей:")
        for i, key in enumerate(missing_keys[:5]):
            print(f"    {i+1}. {key}")
    
    if unexpected_keys:
        print(f"Первые 5 неожиданных ключей:")
        for i, key in enumerate(unexpected_keys[:5]):
            print(f"    {i+1}. {key}")
    
    # Устанавливаем токенизатор
    model.set_tokenizer(tokenizer)
    
    print("✓ Модель создана и веса загружены!")
    
    return model, missing_keys


model, missing_keys = get_custom_hf_model_with_direct_conversion(model_name)

Загружаем модель с прямой конвертацией: ExplosionNuclear/Llama-2.3-3B-Instruct-special-merged-with-19-exp


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

HF конфигурация:
  - hidden_size: 3072
  - num_attention_heads: 24
  - intermediate_size: 8192
  - num_hidden_layers: 28
  - vocab_size: 128258
✅ Установлен rotary_base = 500000.0
HookedTransformer конфигурация:
  - d_model: 3072
  - n_layers: 28
  - n_heads: 24
  - d_mlp: 8192
  - gated_mlp: True
Конвертируем веса через convert_llama_weights...
✓ Веса сконвертированы! Количество ключей: 424
Примеры ключей:
  1. embed.W_E: torch.Size([128258, 3072])
  2. blocks.0.ln1.w: torch.Size([3072])
  3. blocks.0.attn.W_Q: torch.Size([24, 3072, 128])
  4. blocks.0.attn._W_K: torch.Size([8, 3072, 128])
  5. blocks.0.attn._W_V: torch.Size([8, 3072, 128])
Загрузка весов:
  - Отсутствующие ключи: 112
  - Неожиданные ключи: 0
Первые 5 отсутствующих ключей:
    1. blocks.0.attn.mask
    2. blocks.0.attn.IGNORE
    3. blocks.0.attn.rotary_sin
    4. blocks.0.attn.rotary_cos
    5. blocks.1.attn.mask
✓ Модель создана и веса загружены!


In [31]:
def test_model_equivalence_fixed(hf_model, hooked_model, test_text="The capital of France is", max_length=10):
    """
    Исправленный тест эквивалентности моделей
    """
    import torch
    
    print(f"🔍 Тестируем эквивалентность моделей")
    print(f"Тестовый текст: '{test_text}'")
    print("="*60)
    
    tokenizer = hooked_model.tokenizer
    tokens = tokenizer.encode(test_text, return_tensors="pt")
    print(f"Токены: {tokens}")
    
    hf_model.eval()
    hooked_model.eval()
    
    with torch.no_grad():
        print("\n1️⃣ ТЕСТ ЛОГИТОВ")
        print("-" * 30)
        
        hf_outputs = hf_model(tokens)
        hf_logits = hf_outputs.logits
        hooked_logits = hooked_model(tokens)
        
        logits_diff = torch.abs(hf_logits - hooked_logits).max().item()
        print(f"Максимальная разница логитов: {logits_diff:.6f}")
        
        hf_next_token = hf_logits[0, -1].argmax().item()
        hooked_next_token = hooked_logits[0, -1].argmax().item()
        
        print(f"HF предсказание: '{tokenizer.decode([hf_next_token])}' (token {hf_next_token})")
        print(f"Hooked предсказание: '{tokenizer.decode([hooked_next_token])}' (token {hooked_next_token})")
        
        logits_match = hf_next_token == hooked_next_token
        print(f"Предсказания совпадают: {'✅' if logits_match else '❌'}")
        
        print("\n2️⃣ ТЕСТ АКТИВАЦИЙ (УПРОЩЕННЫЙ)")
        print("-" * 30)
        
        # Упрощенная версия - используем hooks для обеих моделей
        def get_hf_activations_simple(layer_idx):
            """
            Получаем активации из HF модели через hooks
            """
            activations = {}
            
            def hf_hook(module, input, output):
                # output может быть tuple или tensor
                if isinstance(output, tuple):
                    activations['hf_layer'] = output[0].clone()
                else:
                    activations['hf_layer'] = output.clone()
            
            # Регистрируем hook на выход нужного слоя
            handle = hf_model.model.layers[layer_idx].register_forward_hook(hf_hook)
            
            try:
                # Прогоняем через модель
                _ = hf_model(tokens)
                return activations.get('hf_layer')
            finally:
                handle.remove()
        
        def get_hooked_activations_simple(layer_idx):
            """
            Получаем активации из HookedTransformer
            """
            activations = {}
            
            def hook_fn(activation, hook):
                activations[hook.name] = activation.clone()
                return activation
            
            hook_name = f"blocks.{layer_idx}.hook_resid_post"
            hooked_model.add_hook(hook_name, hook_fn)
            
            try:
                _ = hooked_model(tokens)
                return activations.get(hook_name)
            finally:
                hooked_model.reset_hooks()
        
        # Тестируем несколько слоев
        layers_to_test = [0, 1, hooked_model.cfg.n_layers - 1]
        activation_results = {}
        
        for layer_idx in layers_to_test:
            print(f"\nТестируем слой {layer_idx}:")
            
            try:
                hf_activations = get_hf_activations_simple(layer_idx)
                hooked_activations = get_hooked_activations_simple(layer_idx)
                
                if hf_activations is not None and hooked_activations is not None:
                    print(f"  HF активации: {hf_activations.shape}")
                    print(f"  Hooked активации: {hooked_activations.shape}")
                    
                    # Проверяем размеры
                    if hf_activations.shape == hooked_activations.shape:
                        act_diff = torch.abs(hf_activations - hooked_activations).max().item()
                        act_mean_diff = torch.abs(hf_activations - hooked_activations).mean().item()
                        
                        print(f"  Максимальная разница: {act_diff:.6f}")
                        print(f"  Средняя разница: {act_mean_diff:.6f}")
                        
                        threshold = 1e-2  # Еще более мягкий порог
                        activation_results[layer_idx] = {
                            'max_diff': act_diff,
                            'mean_diff': act_mean_diff,
                            'match': act_diff < threshold
                        }
                        
                        print(f"  Активации совпадают (порог {threshold}): {'✅' if act_diff < threshold else '❌'}")
                        
                        # Статистика
                        diff_tensor = torch.abs(hf_activations - hooked_activations)
                        print(f"  Медиана: {diff_tensor.median().item():.8f}")
                        print(f"  95-й процентиль: {diff_tensor.quantile(0.95).item():.8f}")
                    else:
                        print(f"  ❌ Размеры не совпадают!")
                        activation_results[layer_idx] = {'match': False}
                else:
                    print(f"  ❌ Не удалось получить активации")
                    activation_results[layer_idx] = {'match': False}
                    
            except Exception as e:
                print(f"  ❌ Ошибка: {e}")
                activation_results[layer_idx] = {'match': False}
        
        print("\n3️⃣ ТЕСТ ПРОМЕЖУТОЧНЫХ СЛОЕВ")
        print("-" * 30)
        
        # Альтернативный способ - сравним embeddings после каждого компонента
        def test_layer_components(layer_idx=0):
            """
            Тестируем компоненты одного слоя
            """
            print(f"Тестируем компоненты слоя {layer_idx}:")
            
            # Получаем активации после attention
            hf_attn_activations = {}
            hooked_attn_activations = {}
            
            def hf_attn_hook(module, input, output):
                if isinstance(output, tuple):
                    hf_attn_activations['attn_out'] = output[0].clone()
                else:
                    hf_attn_activations['attn_out'] = output.clone()
            
            def hooked_attn_hook(activation, hook):
                hooked_attn_activations['attn_out'] = activation.clone()
                return activation
            
            # Hooks для attention
            hf_handle = hf_model.model.layers[layer_idx].self_attn.register_forward_hook(hf_attn_hook)
            hooked_model.add_hook(f"blocks.{layer_idx}.hook_attn_out", hooked_attn_hook)
            
            try:
                _ = hf_model(tokens)
                _ = hooked_model(tokens)
                
                if 'attn_out' in hf_attn_activations and 'attn_out' in hooked_attn_activations:
                    hf_attn = hf_attn_activations['attn_out']
                    hooked_attn = hooked_attn_activations['attn_out']
                    
                    if hf_attn.shape == hooked_attn.shape:
                        attn_diff = torch.abs(hf_attn - hooked_attn).max().item()
                        print(f"  Attention выход: {attn_diff:.6f} {'✅' if attn_diff < 1e-2 else '❌'}")
                    else:
                        print(f"  Attention размеры не совпадают: HF={hf_attn.shape}, Hooked={hooked_attn.shape}")
                
            finally:
                hf_handle.remove()
                hooked_model.reset_hooks()
        
        # Тестируем компоненты первого слоя
        test_layer_components(0)
        
        print("\n🎯 ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
        print("="*60)
        
        activation_matches = sum(1 for r in activation_results.values() if r.get('match', False))
        total_layers = len(activation_results)
        
        print(f"Логиты совпадают: {'✅' if logits_match else '❌'} (разница: {logits_diff:.6f})")
        print(f"Активации совпадают: {activation_matches}/{total_layers} слоев")
        
        # Более мягкие критерии успеха
        reasonable_logits = logits_diff < 0.1
        some_activations_match = activation_matches > 0
        
        print(f"Логиты разумные (< 0.1): {'✅' if reasonable_logits else '❌'}")
        print(f"Хотя бы некоторые активации совпадают: {'✅' if some_activations_match else '❌'}")
        
        overall_success = logits_match and reasonable_logits
        print(f"\n🎉 ОБЩИЙ РЕЗУЛЬТАТ: {'✅ МОДЕЛЬ РАБОТАЕТ КОРРЕКТНО' if overall_success else '❌ ЕСТЬ ПРОБЛЕМЫ'}")
        
        return {
            'logits_match': logits_match,
            'logits_diff': logits_diff,
            'activation_results': activation_results,
            'overall_success': overall_success
        }


In [24]:
def diagnose_attention_weights(hf_model, hooked_model):
    """
    Детальная диагностика attention весов
    """
    print("🔍 ДЕТАЛЬНАЯ ДИАГНОСТИКА ATTENTION ВЕСОВ")
    print("="*60)
    
    hf_state_dict = hf_model.state_dict()
    hooked_state_dict = hooked_model.state_dict()
    
    # Проверяем первый слой
    layer_idx = 0
    
    print(f"\nСлой {layer_idx}:")
    
    # HF веса
    hf_q = hf_state_dict[f'model.layers.{layer_idx}.self_attn.q_proj.weight']
    hf_k = hf_state_dict[f'model.layers.{layer_idx}.self_attn.k_proj.weight'] 
    hf_v = hf_state_dict[f'model.layers.{layer_idx}.self_attn.v_proj.weight']
    hf_o = hf_state_dict[f'model.layers.{layer_idx}.self_attn.o_proj.weight']
    
    # Hooked веса
    hooked_q = hooked_state_dict[f'blocks.{layer_idx}.attn.W_Q']
    
    # Проверяем есть ли GQA веса
    if f'blocks.{layer_idx}.attn._W_K' in hooked_state_dict:
        hooked_k = hooked_state_dict[f'blocks.{layer_idx}.attn._W_K']
        hooked_v = hooked_state_dict[f'blocks.{layer_idx}.attn._W_V']
        print("  Используется GQA (Grouped Query Attention)")
    else:
        hooked_k = hooked_state_dict[f'blocks.{layer_idx}.attn.W_K']
        hooked_v = hooked_state_dict[f'blocks.{layer_idx}.attn.W_V']
        print("  Используется обычный attention")
    
    hooked_o = hooked_state_dict[f'blocks.{layer_idx}.attn.W_O']
    
    print(f"\nРазмеры весов:")
    print(f"  HF: Q={hf_q.shape}, K={hf_k.shape}, V={hf_v.shape}, O={hf_o.shape}")
    print(f"  Hooked: Q={hooked_q.shape}, K={hooked_k.shape}, V={hooked_v.shape}, O={hooked_o.shape}")
    
    # Конфигурация модели
    n_heads = hooked_model.cfg.n_heads
    d_head = hooked_model.cfg.d_head  
    d_model = hooked_model.cfg.d_model
    n_kv_heads = getattr(hooked_model.cfg, 'n_key_value_heads', n_heads)
    
    print(f"\nКонфигурация:")
    print(f"  n_heads: {n_heads}")
    print(f"  n_kv_heads: {n_kv_heads}")
    print(f"  d_head: {d_head}")
    print(f"  d_model: {d_model}")
    
    # Проверяем правильность reshape для Q
    print(f"\nПроверка Q weights:")
    expected_hf_q_shape = (n_heads * d_head, d_model)
    print(f"  Ожидаемый HF размер: {expected_hf_q_shape}")
    print(f"  Фактический HF размер: {hf_q.shape}")
    print(f"  Ожидаемый Hooked размер: ({n_heads}, {d_model}, {d_head})")
    print(f"  Фактический Hooked размер: {hooked_q.shape}")
    
    if hf_q.shape == expected_hf_q_shape:
        # Правильный reshape для Q
        hf_q_reshaped = hf_q.view(n_heads, d_head, d_model).transpose(1, 2)
        q_diff = torch.abs(hf_q_reshaped - hooked_q).max().item()
        print(f"  Q разница после reshape: {q_diff:.8f} {'✅' if q_diff < 1e-6 else '❌'}")
    else:
        print(f"  ❌ Неожиданный размер Q весов")
    
    # Проверяем K и V (с учетом GQA)
    print(f"\nПроверка K weights:")
    expected_hf_k_shape = (n_kv_heads * d_head, d_model)
    print(f"  Ожидаемый HF размер: {expected_hf_k_shape}")
    print(f"  Фактический HF размер: {hf_k.shape}")
    print(f"  Ожидаемый Hooked размер: ({n_kv_heads}, {d_model}, {d_head})")
    print(f"  Фактический Hooked размер: {hooked_k.shape}")
    
    if hf_k.shape == expected_hf_k_shape:
        hf_k_reshaped = hf_k.view(n_kv_heads, d_head, d_model).transpose(1, 2)
        k_diff = torch.abs(hf_k_reshaped - hooked_k).max().item()
        print(f"  K разница после reshape: {k_diff:.8f} {'✅' if k_diff < 1e-6 else '❌'}")
    else:
        print(f"  ❌ Неожиданный размер K весов")
    
    # Проверяем O
    print(f"\nПроверка O weights:")
    expected_hf_o_shape = (d_model, n_heads * d_head)
    print(f"  Ожидаемый HF размер: {expected_hf_o_shape}")
    print(f"  Фактический HF размер: {hf_o.shape}")
    print(f"  Ожидаемый Hooked размер: ({n_heads}, {d_head}, {d_model})")
    print(f"  Фактический Hooked размер: {hooked_o.shape}")
    
    if hf_o.shape == expected_hf_o_shape:
        hf_o_reshaped = hf_o.T.view(n_heads, d_head, d_model)
        o_diff = torch.abs(hf_o_reshaped - hooked_o).max().item()
        print(f"  O разница после reshape: {o_diff:.8f} {'✅' if o_diff < 1e-6 else '❌'}")
    else:
        print(f"  ❌ Неожиданный размер O весов")

def diagnose_mlp_weights(hf_model, hooked_model):
    """
    Детальная диагностика MLP весов
    """
    print("\n🔍 ДЕТАЛЬНАЯ ДИАГНОСТИКА MLP ВЕСОВ")
    print("="*60)
    
    hf_state_dict = hf_model.state_dict()
    hooked_state_dict = hooked_model.state_dict()
    
    layer_idx = 0
    
    # HF MLP веса
    hf_gate = hf_state_dict[f'model.layers.{layer_idx}.mlp.gate_proj.weight']
    hf_up = hf_state_dict[f'model.layers.{layer_idx}.mlp.up_proj.weight']
    hf_down = hf_state_dict[f'model.layers.{layer_idx}.mlp.down_proj.weight']
    
    # Hooked MLP веса
    hooked_gate = hooked_state_dict[f'blocks.{layer_idx}.mlp.W_gate']
    hooked_in = hooked_state_dict[f'blocks.{layer_idx}.mlp.W_in']
    hooked_out = hooked_state_dict[f'blocks.{layer_idx}.mlp.W_out']
    
    print(f"Размеры MLP весов:")
    print(f"  HF: gate={hf_gate.shape}, up={hf_up.shape}, down={hf_down.shape}")
    print(f"  Hooked: gate={hooked_gate.shape}, in={hooked_in.shape}, out={hooked_out.shape}")
    
    d_model = hooked_model.cfg.d_model
    d_mlp = hooked_model.cfg.d_mlp
    
    print(f"\nОжидаемые размеры:")
    print(f"  HF: gate=({d_mlp}, {d_model}), up=({d_mlp}, {d_model}), down=({d_model}, {d_mlp})")
    print(f"  Hooked: gate=({d_model}, {d_mlp}), in=({d_model}, {d_mlp}), out=({d_mlp}, {d_model})")
    
    # Проверяем соответствие
    gate_diff = torch.abs(hf_gate.T - hooked_gate).max().item()
    in_diff = torch.abs(hf_up.T - hooked_in).max().item()  
    out_diff = torch.abs(hf_down.T - hooked_out).max().item()
    
    print(f"\nРазности после транспозиции:")
    print(f"  Gate (gate_proj -> W_gate): {gate_diff:.8f} {'✅' if gate_diff < 1e-6 else '❌'}")
    print(f"  In (up_proj -> W_in): {in_diff:.8f} {'✅' if in_diff < 1e-6 else '❌'}")
    print(f"  Out (down_proj -> W_out): {out_diff:.8f} {'✅' if out_diff < 1e-6 else '❌'}")

# Запускаем диагностику
diagnose_attention_weights(hf_model, hooked_model)
diagnose_mlp_weights(hf_model, hooked_model)

🔍 ДЕТАЛЬНАЯ ДИАГНОСТИКА ATTENTION ВЕСОВ

Слой 0:
  Используется GQA (Grouped Query Attention)

Размеры весов:
  HF: Q=torch.Size([3072, 3072]), K=torch.Size([1024, 3072]), V=torch.Size([1024, 3072]), O=torch.Size([3072, 3072])
  Hooked: Q=torch.Size([24, 3072, 128]), K=torch.Size([8, 3072, 128]), V=torch.Size([8, 3072, 128]), O=torch.Size([24, 128, 3072])

Конфигурация:
  n_heads: 24
  n_kv_heads: 8
  d_head: 128
  d_model: 3072

Проверка Q weights:
  Ожидаемый HF размер: (3072, 3072)
  Фактический HF размер: torch.Size([3072, 3072])
  Ожидаемый Hooked размер: (24, 3072, 128)
  Фактический Hooked размер: torch.Size([24, 3072, 128])
  Q разница после reshape: 0.00000000 ✅

Проверка K weights:
  Ожидаемый HF размер: (1024, 3072)
  Фактический HF размер: torch.Size([1024, 3072])
  Ожидаемый Hooked размер: (8, 3072, 128)
  Фактический Hooked размер: torch.Size([8, 3072, 128])
  K разница после reshape: 0.00000000 ✅

Проверка O weights:
  Ожидаемый HF размер: (3072, 3072)
  Фактический HF р

In [23]:
def diagnose_model_behavior(hf_model, hooked_model):
    """
    Диагностика поведения модели - возможные причины расхождений
    """
    print("🔍 ДИАГНОСТИКА ПОВЕДЕНИЯ МОДЕЛИ")
    print("="*60)
    
    # Простые токены для тестирования
    test_text = "Hello"
    tokenizer = hooked_model.tokenizer
    tokens = tokenizer.encode(test_text, return_tensors="pt")
    
    print(f"Тестовый текст: '{test_text}'")
    print(f"Токены: {tokens}")
    
    hf_model.eval()
    hooked_model.eval()
    
    with torch.no_grad():
        print("\n1️⃣ ПРОВЕРКА EMBEDDINGS")
        print("-" * 30)
        
        # Получаем embeddings
        hf_embed_out = hf_model.model.embed_tokens(tokens)
        hooked_embed_out = hooked_model.embed(tokens)
        
        embed_diff = torch.abs(hf_embed_out - hooked_embed_out).max().item()
        print(f"Embeddings разница: {embed_diff:.8f} {'✅' if embed_diff < 1e-6 else '❌'}")
        
        print("\n2️⃣ ПРОВЕРКА ПОЗИЦИОННЫХ EMBEDDINGS")
        print("-" * 30)
        
        # HookedTransformer использует rotary embeddings, HF тоже
        # Но могут быть различия в реализации
        print("HF модель использует RoPE (Rotary Position Embedding)")
        print("HookedTransformer тоже использует RoPE")
        
        # Проверим конфигурацию RoPE
        hf_config = hf_model.config
        hooked_config = hooked_model.cfg
        
        print(f"HF rope_theta: {getattr(hf_config, 'rope_theta', 'не указан')}")
        print(f"HF max_position_embeddings: {hf_config.max_position_embeddings}")
        print(f"Hooked rotary_dim: {hooked_config.rotary_dim}")
        print(f"Hooked n_ctx: {hooked_config.n_ctx}")
        
        # Проверим base для rotary embeddings
        if hasattr(hf_config, 'rope_theta'):
            rope_theta_match = getattr(hooked_config, 'rotary_base', 10000.0) == hf_config.rope_theta
            print(f"RoPE base совпадает: {'✅' if rope_theta_match else '❌'}")
        
        print("\n3️⃣ ПРОВЕРКА НОРМАЛИЗАЦИИ")
        print("-" * 30)
        
        # Проверим RMS norm
        print(f"HF rms_norm_eps: {getattr(hf_config, 'rms_norm_eps', 'не указан')}")
        print(f"Hooked eps: {hooked_config.eps}")
        
        eps_match = hooked_config.eps == getattr(hf_config, 'rms_norm_eps', 1e-6)
        print(f"Epsilon совпадает: {'✅' if eps_match else '❌'}")
        
        print("\n4️⃣ ПРОВЕРКА АКТИВАЦИОННЫХ ФУНКЦИЙ")
        print("-" * 30)
        
        print(f"HF hidden_act: {hf_config.hidden_act}")
        print(f"Hooked act_fn: {hooked_config.act_fn}")
        
        act_match = hooked_config.act_fn == hf_config.hidden_act
        print(f"Активации совпадают: {'✅' if act_match else '❌'}")
        
        print("\n5️⃣ ТЕСТ ПРОМЕЖУТОЧНЫХ АКТИВАЦИЙ")
        print("-" * 30)
        
        # Получим активации после первого слоя
        activations_hf = {}
        activations_hooked = {}
        
        def hf_hook(module, input, output):
            activations_hf['layer_0'] = output[0].clone()
        
        def hooked_hook(activation, hook):
            activations_hooked['layer_0'] = activation.clone()
            return activation
        
        # Добавляем hooks
        hf_handle = hf_model.model.layers[0].register_forward_hook(hf_hook)
        hooked_model.add_hook("blocks.0.hook_resid_post", hooked_hook)
        
        try:
            # Прогоняем через модели
            _ = hf_model(tokens)
            _ = hooked_model(tokens)
            
            if 'layer_0' in activations_hf and 'layer_0' in activations_hooked:
                layer_diff = torch.abs(activations_hf['layer_0'] - activations_hooked['layer_0']).max().item()
                print(f"Активации после слоя 0: {layer_diff:.8f} {'✅' if layer_diff < 1e-6 else '❌'}")
                
                # Статистика различий
                diff_tensor = torch.abs(activations_hf['layer_0'] - activations_hooked['layer_0'])
                print(f"  Средняя разница: {diff_tensor.mean().item():.8f}")
                print(f"  Медианная разница: {diff_tensor.median().item():.8f}")
                print(f"  90-й процентиль: {diff_tensor.quantile(0.9).item():.8f}")
            else:
                print("❌ Не удалось получить активации")
        
        finally:
            hf_handle.remove()
            hooked_model.reset_hooks()
        
        print("\n6️⃣ ПРОВЕРКА ЧИСЛЕННОЙ СТАБИЛЬНОСТИ")
        print("-" * 30)
        
        # Проверим на разных входах
        test_cases = [
            "A",
            "The", 
            "Hello world",
            "This is a test"
        ]
        
        max_diffs = []
        
        for test_case in test_cases:
            tokens_test = tokenizer.encode(test_case, return_tensors="pt")
            
            hf_out = hf_model(tokens_test).logits
            hooked_out = hooked_model(tokens_test)
            
            diff = torch.abs(hf_out - hooked_out).max().item()
            max_diffs.append(diff)
            
            print(f"  '{test_case}': макс. разница = {diff:.6f}")
        
        avg_diff = sum(max_diffs) / len(max_diffs)
        print(f"\nСредняя максимальная разница: {avg_diff:.6f}")
        
        # Проверим, растет ли ошибка с длиной последовательности
        print(f"Зависимость от длины: {'❌ Есть' if max(max_diffs) > min(max_diffs) * 2 else '✅ Нет'}")

# Запускаем диагностику
diagnose_model_behavior(hf_model, hooked_model)

🔍 ДИАГНОСТИКА ПОВЕДЕНИЯ МОДЕЛИ
Тестовый текст: 'Hello'
Токены: tensor([[128000,   9906]])

1️⃣ ПРОВЕРКА EMBEDDINGS
------------------------------
Embeddings разница: 0.00000000 ✅

2️⃣ ПРОВЕРКА ПОЗИЦИОННЫХ EMBEDDINGS
------------------------------
HF модель использует RoPE (Rotary Position Embedding)
HookedTransformer тоже использует RoPE
HF rope_theta: 500000.0
HF max_position_embeddings: 131072
Hooked rotary_dim: 128
Hooked n_ctx: 2048
RoPE base совпадает: ✅

3️⃣ ПРОВЕРКА НОРМАЛИЗАЦИИ
------------------------------
HF rms_norm_eps: 1e-05
Hooked eps: 1e-05
Epsilon совпадает: ✅

4️⃣ ПРОВЕРКА АКТИВАЦИОННЫХ ФУНКЦИЙ
------------------------------
HF hidden_act: silu
Hooked act_fn: silu
Активации совпадают: ✅

5️⃣ ТЕСТ ПРОМЕЖУТОЧНЫХ АКТИВАЦИЙ
------------------------------
Активации после слоя 0: 0.00007771 ❌
  Средняя разница: 0.00000311
  Медианная разница: 0.00000019
  90-й процентиль: 0.00000944

6️⃣ ПРОВЕРКА ЧИСЛЕННОЙ СТАБИЛЬНОСТИ
------------------------------
  'A': макс. разница = 0

In [22]:
hooked_model = model

In [15]:
def get_custom_hf_model_with_direct_conversion(model_name: str, kwargs: Dict[str, Any] = {}) -> HookedTransformer:
    """
    Напрямую используем convert_llama_weights, обходя проверку официальных моделей
    """
    print(f"Загружаем модель с прямой конвертацией: {model_name}")
    
    # Загружаем HF модель
    hf_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="cpu",
        **kwargs
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    hf_config = hf_model.config
    
    print(f"HF конфигурация:")
    print(f"  - hidden_size: {hf_config.hidden_size}")
    print(f"  - num_attention_heads: {hf_config.num_attention_heads}")
    print(f"  - intermediate_size: {hf_config.intermediate_size}")
    print(f"  - num_hidden_layers: {hf_config.num_hidden_layers}")
    print(f"  - vocab_size: {hf_config.vocab_size}")
    
    # Создаем конфигурацию как для Llama
    cfg_dict = {
        "d_model": hf_config.hidden_size,
        "d_head": hf_config.hidden_size // hf_config.num_attention_heads,
        "n_heads": hf_config.num_attention_heads,
        "d_mlp": hf_config.intermediate_size,
        "n_layers": hf_config.num_hidden_layers,
        "n_ctx": min(hf_config.max_position_embeddings, 2048),
        "eps": getattr(hf_config, 'rms_norm_eps', 1e-6),
        "d_vocab": hf_config.vocab_size,
        "act_fn": hf_config.hidden_act,
        "normalization_type": "RMS",
        "positional_embedding_type": "rotary",
        "rotary_adjacent_pairs": False,
        "rotary_dim": hf_config.hidden_size // hf_config.num_attention_heads,
        "final_rms": True,
        "gated_mlp": True,
        "model_name": model_name.split("/")[-1],
        "init_weights": False,
        "device": "cpu",
        "dtype": torch.float32,
    }
    
    # Добавляем n_key_value_heads если есть
    if hasattr(hf_config, 'num_key_value_heads') and hf_config.num_key_value_heads != hf_config.num_attention_heads:
        cfg_dict["n_key_value_heads"] = hf_config.num_key_value_heads
    
    cfg = HookedTransformerConfig.from_dict(cfg_dict)
    
    print(f"HookedTransformer конфигурация:")
    print(f"  - d_model: {cfg.d_model}")
    print(f"  - n_layers: {cfg.n_layers}")
    print(f"  - n_heads: {cfg.n_heads}")
    print(f"  - d_mlp: {cfg.d_mlp}")
    print(f"  - gated_mlp: {cfg.gated_mlp}")
    
    # Импортируем и используем convert_llama_weights напрямую
    from transformer_lens.loading_from_pretrained import convert_llama_weights
    
    print("Конвертируем веса через convert_llama_weights...")
    
    # Отключаем градиенты для HF модели
    for param in hf_model.parameters():
        param.requires_grad = False
    
    # Конвертируем веса
    state_dict = convert_llama_weights(hf_model, cfg)
    
    print(f"✓ Веса сконвертированы! Количество ключей: {len(state_dict)}")
    print(f"Примеры ключей:")
    for i, key in enumerate(list(state_dict.keys())[:5]):
        print(f"  {i+1}. {key}: {state_dict[key].shape}")
    
    # Создаем HookedTransformer
    model = HookedTransformer(cfg)
    
    # Загружаем веса
    missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
    
    print(f"Загрузка весов:")
    print(f"  - Отсутствующие ключи: {len(missing_keys)}")
    print(f"  - Неожиданные ключи: {len(unexpected_keys)}")
    
    if missing_keys:
        print(f"Первые 5 отсутствующих ключей:")
        for i, key in enumerate(missing_keys[:5]):
            print(f"    {i+1}. {key}")
    
    if unexpected_keys:
        print(f"Первые 5 неожиданных ключей:")
        for i, key in enumerate(unexpected_keys[:5]):
            print(f"    {i+1}. {key}")
    
    # Устанавливаем токенизатор
    model.set_tokenizer(tokenizer)
    
    print("✓ Модель создана и веса загружены!")
    
    return model


model = get_custom_hf_model_with_pretrained_config(model_name)

Загружаем модель через встроенную систему transformer_lens: ExplosionNuclear/Llama-2.3-3B-Instruct-special-merged-with-19-exp


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Конфигурация создана:
  - d_model: 3072
  - n_layers: 28
  - n_heads: 24
  - d_mlp: 8192
  - gated_mlp: True
  - original_architecture: LlamaForCausalLM
Конвертируем веса через встроенную систему...


ValueError: ExplosionNuclear/Llama-2.3-3B-Instruct-special-merged-with-19-exp not found. Valid official model names (excl aliases): ['gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl', 'distilgpt2', 'facebook/opt-125m', 'facebook/opt-1.3b', 'facebook/opt-2.7b', 'facebook/opt-6.7b', 'facebook/opt-13b', 'facebook/opt-30b', 'facebook/opt-66b', 'EleutherAI/gpt-neo-125M', 'EleutherAI/gpt-neo-1.3B', 'EleutherAI/gpt-neo-2.7B', 'EleutherAI/gpt-j-6B', 'EleutherAI/gpt-neox-20b', 'stanford-crfm/alias-gpt2-small-x21', 'stanford-crfm/battlestar-gpt2-small-x49', 'stanford-crfm/caprica-gpt2-small-x81', 'stanford-crfm/darkmatter-gpt2-small-x343', 'stanford-crfm/expanse-gpt2-small-x777', 'stanford-crfm/arwen-gpt2-medium-x21', 'stanford-crfm/beren-gpt2-medium-x49', 'stanford-crfm/celebrimbor-gpt2-medium-x81', 'stanford-crfm/durin-gpt2-medium-x343', 'stanford-crfm/eowyn-gpt2-medium-x777', 'EleutherAI/pythia-14m', 'EleutherAI/pythia-31m', 'EleutherAI/pythia-70m', 'EleutherAI/pythia-160m', 'EleutherAI/pythia-4

In [8]:
# Шаг 6: Тестируем модель с простым примером
print("6. Тестируем созданную модель...")

try:
    # Устанавливаем токенизатор
    hooked_model.set_tokenizer(tokenizer)
    
    # Простой тест
    test_text = "The capital of France is"
    print(f"Тестовый текст: '{test_text}'")
    
    # Токенизируем
    tokens = hooked_model.to_tokens(test_text)
    print(f"Токены: {tokens}")
    print(f"Размер токенов: {tokens.shape}")
    
    # Прогоняем через модель
    with torch.no_grad():
        logits = hooked_model(tokens)
    
    print(f"✓ Модель работает!")
    print(f"Размер выходных логитов: {logits.shape}")
    
    # Получаем предсказание
    next_token = logits[0, -1].argmax().item()
    next_word = tokenizer.decode([next_token])
    print(f"Предсказанное следующее слово: '{next_word}'")
    
    # Проверяем активации на нужном hook_point
    hook_point = "blocks.0.hook_resid_pre"  # Из конфигурации try.yaml
    
    def capture_activations(activations, hook):
        print(f"Активации на {hook.name}: {activations.shape}")
        return activations
    
    print(f"\n7. Проверяем hook_point: {hook_point}")
    hooked_model.add_hook(hook_point, capture_activations)
    
    with torch.no_grad():
        _ = hooked_model(tokens)
    
    hooked_model.reset_hooks()
    print(f"✓ Hook point работает корректно!")
    
except Exception as e:
    print(f"✗ Ошибка при тестировании модели: {e}")
    import traceback
    traceback.print_exc()


6. Тестируем созданную модель...
Тестовый текст: 'The capital of France is'
Токены: tensor([[128000,    791,   6864,    315,   9822,    374]])
Размер токенов: torch.Size([1, 6])
✓ Модель работает!
Размер выходных логитов: torch.Size([1, 6, 128258])
Предсказанное следующее слово: ' hj'

7. Проверяем hook_point: blocks.0.hook_resid_pre
Активации на blocks.0.hook_resid_pre: torch.Size([1, 6, 3072])
✓ Hook point работает корректно!


# Исправленная версия создания модели

Основываясь на диагностике выше, создадим улучшенную версию функции `get_custom_hf_model`


In [ ]:
def get_custom_hf_model_improved(model_name: str, kwargs: Dict[str, Any] = {}) -> HookedTransformer:
    """
    Улучшенная версия создания HookedTransformer из HuggingFace модели
    с более детальной диагностикой и обработкой ошибок
    """
    print(f"Загружаем модель: {model_name}")
    
    # Загружаем HF модель
    hf_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,  # Явно указываем dtype
        **kwargs
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    hf_config = hf_model.config
    
    print(f"Конфигурация модели:")
    print(f"  - model_type: {hf_config.model_type}")
    print(f"  - num_hidden_layers: {hf_config.num_hidden_layers}")
    print(f"  - hidden_size: {hf_config.hidden_size}")
    print(f"  - num_attention_heads: {hf_config.num_attention_heads}")
    print(f"  - intermediate_size: {hf_config.intermediate_size}")
    
    # Проверяем особенности конкретной модели
    max_ctx = min(hf_config.max_position_embeddings, 2048)
    
    # Определяем тип нормализации на основе конфигурации
    normalization_type = "RMS"  # По умолчанию для Llama
    if hasattr(hf_config, 'rms_norm_eps'):
        normalization_type = "RMS"
    elif hasattr(hf_config, 'layer_norm_eps'):
        normalization_type = "LN"
    
    # Создаем конфигурацию HookedTransformer
    cfg = HookedTransformerConfig(
        n_layers=hf_config.num_hidden_layers,
        d_model=hf_config.hidden_size,
        d_head=hf_config.hidden_size // hf_config.num_attention_heads,
        n_heads=hf_config.num_attention_heads,
        d_mlp=hf_config.intermediate_size,
        d_vocab=hf_config.vocab_size,
        n_ctx=max_ctx,
        act_fn=hf_config.hidden_act,
        model_name=model_name,
        normalization_type=normalization_type,
        device="cpu",
        use_hook_mlp_in=True,
        # Дополнительные параметры для совместимости
        eps=getattr(hf_config, 'rms_norm_eps', getattr(hf_config, 'layer_norm_eps', 1e-5)),
    )
    
    print(f"Создаем HookedTransformer...")
    model = HookedTransformer(cfg)
    
    print(f"Загружаем веса...")
    # Загружаем веса с более детальной диагностикой
    missing_keys, unexpected_keys = model.load_state_dict(hf_model.state_dict(), strict=False)
    
    print(f"  - Отсутствующие ключи: {len(missing_keys)}")
    print(f"  - Неожиданные ключи: {len(unexpected_keys)}")
    
    if len(missing_keys) > 0:
        print("  Первые отсутствующие ключи:")
        for key in missing_keys[:3]:
            print(f"    - {key}")
    
    # Устанавливаем токенизатор
    model.set_tokenizer(tokenizer)
    
    print(f"✓ Модель создана успешно")
    return model

# Тестируем улучшенную версию
print("="*50)
print("ТЕСТИРУЕМ УЛУЧШЕННУЮ ВЕРСИЮ")
print("="*50)

improved_model = get_custom_hf_model_improved(model_name)


In [ ]:
# Тестируем улучшенную модель
print("8. Тестируем улучшенную модель...")

try:
    # Простой тест генерации
    test_text = "The capital of France is"
    print(f"Тестовый текст: '{test_text}'")
    
    tokens = improved_model.to_tokens(test_text)
    print(f"Токены: {tokens.shape}")
    
    with torch.no_grad():
        logits = improved_model(tokens)
    
    print(f"✓ Модель работает! Логиты: {logits.shape}")
    
    # Тестируем hook points из конфигурации
    hook_points_to_test = [
        "blocks.0.hook_resid_pre",
        "blocks.1.hook_resid_pre", 
        "blocks.2.hook_resid_pre",
        "blocks.3.hook_resid_pre",
        "blocks.4.hook_resid_pre"
    ]
    
    print(f"\n9. Тестируем hook points из try.yaml...")
    
    activations_captured = {}
    
    def capture_hook(name):
        def hook_fn(activations, hook):
            activations_captured[name] = activations.shape
            return activations
        return hook_fn
    
    # Добавляем хуки
    for hook_point in hook_points_to_test:
        improved_model.add_hook(hook_point, capture_hook(hook_point))
    
    # Прогоняем через модель
    with torch.no_grad():
        _ = improved_model(tokens)
    
    # Выводим результаты
    print("Активации на hook points:")
    for hook_point in hook_points_to_test:
        if hook_point in activations_captured:
            print(f"  ✓ {hook_point}: {activations_captured[hook_point]}")
        else:
            print(f"  ✗ {hook_point}: НЕ НАЙДЕН")
    
    improved_model.reset_hooks()
    
    # Проверяем соответствие d_in из конфигурации
    expected_d_in = 3072  # Из try.yaml
    actual_d_model = improved_model.cfg.d_model
    
    print(f"\n10. Проверяем соответствие конфигурации:")
    print(f"  - Ожидаемый d_in: {expected_d_in}")
    print(f"  - Фактический d_model: {actual_d_model}")
    print(f"  - Совпадает: {expected_d_in == actual_d_model}")
    
    if expected_d_in == actual_d_model:
        print("✅ МОДЕЛЬ СОЗДАНА КОРРЕКТНО!")
    else:
        print("❌ НЕСООТВЕТСТВИЕ РАЗМЕРНОСТЕЙ!")
        
except Exception as e:
    print(f"✗ Ошибка при тестировании улучшенной модели: {e}")
    import traceback
    traceback.print_exc()


# Финальная версия для замены в utils.py

Если диагностика показала проблемы, вот исправленная версия функции для замены в `sae_training/utils.py`:


In [ ]:
def get_custom_hf_model_fixed(model_name: str, kwargs: Dict[str, Any] = {}) -> HookedTransformer:
    """
    Исправленная версия get_custom_hf_model для корректного создания HookedTransformer
    из HuggingFace моделей, включая Llama 3.2 3B
    """
    import logging
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    
    logger.info(f"Загружаем HuggingFace модель: {model_name}")
    
    # Загружаем модель с явным указанием типа данных
    hf_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,  # Явно указываем float32
        device_map="cpu",  # Загружаем на CPU сначала
        **kwargs
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    hf_config = hf_model.config
    
    # Логируем ключевые параметры
    logger.info(f"Конфигурация модели:")
    logger.info(f"  - model_type: {hf_config.model_type}")
    logger.info(f"  - hidden_size (d_model): {hf_config.hidden_size}")
    logger.info(f"  - num_attention_heads: {hf_config.num_attention_heads}")
    logger.info(f"  - num_hidden_layers: {hf_config.num_hidden_layers}")
    logger.info(f"  - intermediate_size (d_mlp): {hf_config.intermediate_size}")
    logger.info(f"  - vocab_size: {hf_config.vocab_size}")
    logger.info(f"  - max_position_embeddings: {hf_config.max_position_embeddings}")
    logger.info(f"  - hidden_act: {hf_config.hidden_act}")
    
    # Ограничиваем размер контекста для экономии памяти
    max_ctx = min(hf_config.max_position_embeddings, 2048)
    
    # Определяем тип нормализации
    normalization_type = "RMS"  # По умолчанию для Llama
    eps = 1e-5  # Значение по умолчанию
    
    if hasattr(hf_config, 'rms_norm_eps'):
        normalization_type = "RMS"
        eps = hf_config.rms_norm_eps
    elif hasattr(hf_config, 'layer_norm_eps'):
        normalization_type = "LN"
        eps = hf_config.layer_norm_eps
    
    # Проверяем корректность расчета d_head
    d_head = hf_config.hidden_size // hf_config.num_attention_heads
    if hf_config.hidden_size % hf_config.num_attention_heads != 0:
        raise ValueError(f"hidden_size ({hf_config.hidden_size}) не делится нацело на num_attention_heads ({hf_config.num_attention_heads})")
    
    logger.info(f"Создаем HookedTransformerConfig с параметрами:")
    logger.info(f"  - d_model: {hf_config.hidden_size}")
    logger.info(f"  - d_head: {d_head}")
    logger.info(f"  - n_heads: {hf_config.num_attention_heads}")
    logger.info(f"  - d_mlp: {hf_config.intermediate_size}")
    logger.info(f"  - normalization_type: {normalization_type}")
    logger.info(f"  - eps: {eps}")
    
    # Создаем конфигурацию HookedTransformer
    cfg = HookedTransformerConfig(
        n_layers=hf_config.num_hidden_layers,
        d_model=hf_config.hidden_size,
        d_head=d_head,
        n_heads=hf_config.num_attention_heads,
        d_mlp=hf_config.intermediate_size,
        d_vocab=hf_config.vocab_size,
        n_ctx=max_ctx,
        act_fn=hf_config.hidden_act,
        model_name=model_name,
        normalization_type=normalization_type,
        device="cpu",
        use_hook_mlp_in=True,
        eps=eps,
        # Дополнительные параметры для совместимости с Llama
        final_rms_norm=True if normalization_type == "RMS" else False,
        gated_mlp=True if hasattr(hf_config, 'hidden_act') and 'glu' in hf_config.hidden_act.lower() else False,
    )
    
    logger.info("Создаем HookedTransformer...")
    model = HookedTransformer(cfg)
    
    logger.info("Загружаем веса из HuggingFace модели...")
    
    # Загружаем веса с обработкой ошибок
    try:
        missing_keys, unexpected_keys = model.load_state_dict(hf_model.state_dict(), strict=False)
        
        logger.info(f"Загрузка весов завершена:")
        logger.info(f"  - Отсутствующие ключи: {len(missing_keys)}")
        logger.info(f"  - Неожиданные ключи: {len(unexpected_keys)}")
        
        # Логируем критичные отсутствующие ключи
        critical_missing = [k for k in missing_keys if not any(skip in k for skip in ['lm_head', 'embed_tokens.weight'])]
        if critical_missing:
            logger.warning(f"Критичные отсутствующие ключи: {critical_missing[:5]}")
            
    except Exception as e:
        logger.error(f"Ошибка при загрузке весов: {e}")
        raise
    
    # Устанавливаем токенизатор
    model.set_tokenizer(tokenizer)
    
    # Проверяем работоспособность модели
    try:
        test_tokens = model.to_tokens("Test")
        with torch.no_grad():
            _ = model(test_tokens)
        logger.info("✓ Модель прошла базовый тест")
    except Exception as e:
        logger.error(f"Модель не прошла базовый тест: {e}")
        raise
    
    logger.info("✓ HookedTransformer создан успешно")
    return model

print("Исправленная функция готова для замены в utils.py")
